# 69 Optode co-registration visual check

Visual sanity check for notebook 68 (`coreg_invariance.csv`). For one
subject, run `ColoredStickerProcessor` on the original and the
anonymized scan, project sticker centres along the surface normal to
the scalp (subtract `OPTODE_LENGTH`), match the two centre clouds by
KDTree nearest neighbour, and render everything in two linked PyVista
panels.

The detection, projection, and matching code is identical to notebook
68; only the rendering is new. The deviation stats printed in cell 5
should match the corresponding row of `coreg_invariance.csv` exactly.
If they do, the side-by-side plot tells you what those numbers mean
geometrically.


In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))

import numpy as np
import pyvista as pv
from scipy.spatial import KDTree

import cedalion
import cedalion.vis.blocks as vbx
from cedalion.geometry.landmarks import normalize_landmarks_labels
from cedalion.geometry.photogrammetry.processors import ColoredStickerProcessor

from _thesis_data import load_raw, load_anon, load_landmarks

pv.set_jupyter_backend('server')

SUBJECT = 17
COLORS = {'O': (0.11, 0.21, 0.7, 1.0)}
OPTODE_LENGTH = 22.6 * cedalion.units.mm
L = float(OPTODE_LENGTH.to('mm').magnitude)


## 1. Load surfaces and landmarks

Subject17 is the post-dev-merge verified subject. Change `SUBJECT` in
the previous cell to re-run on any other optode-cohort subject
(16, 18, 19, 20, 21, 22).


In [2]:
surface_orig = load_raw(SUBJECT)
surface_anon = load_anon(SUBJECT)
landmarks = normalize_landmarks_labels(load_landmarks(SUBJECT))

print(f'Subject{SUBJECT}')
print(f'  original  vertices: {len(surface_orig.mesh.vertices):,}')
print(f'  anonymized vertices: {len(surface_anon.mesh.vertices):,}')
print(f'  landmarks: {list(landmarks.label.values)}')


Subject17
  original  vertices: 1,284,667
  anonymized vertices: 593,963
  landmarks: [np.str_('Nz'), np.str_('Iz'), np.str_('Cz'), np.str_('LPA'), np.str_('RPA')]


## 2. Detect stickers on both scans

Same `ColoredStickerProcessor` configuration as notebook 41 and
notebook 68. `centres` carry units; convert to numpy for the matching
step below.


In [3]:
def detect_stickers(surface):
    proc = ColoredStickerProcessor(colors=COLORS)
    centres, normals = proc.process(surface, details=False)
    c_np = centres.pint.dequantify().values
    n_np = normals.values if hasattr(normals, 'values') else np.asarray(normals)
    return centres, normals, c_np, n_np

centres_orig, normals_orig, c_orig, n_orig = detect_stickers(surface_orig)
centres_anon, normals_anon, c_anon, n_anon = detect_stickers(surface_anon)

scalp_orig_np = c_orig - L * n_orig
scalp_anon_np = c_anon - L * n_anon

scalp_orig = centres_orig.copy()
scalp_orig.values = scalp_orig_np
scalp_anon = centres_anon.copy()
scalp_anon.values = scalp_anon_np

print(f'stickers detected -- original: {len(c_orig)}, anonymized: {len(c_anon)}')


[[-62.126129 379.344299 496.446136]
 [-60.2271   380.594482 495.111847]
 [-60.2271   380.594482 495.111847]
 ...
 [ 54.036041  -0.995697 444.60611 ]
 [ -6.470909 201.673172 444.291443]
 [ 63.561325  90.3022   582.236816]]
[[173.067749 152.878815 524.397705]
 [171.551178 156.72226  523.730469]
 [195.392883 145.783371 477.122894]
 ...
 [168.050629   2.322266 444.530457]
 [168.193817   2.722275 444.530457]
 [192.774017  53.522278 444.530457]]
O (0.11, 0.21, 0.7, 1.0)
[0.4250353  1.35472219 1.7145254  0.39138499 1.29862752 0.62320719
 0.38994548 0.61920156 0.78096456 0.49167592 1.33193309 0.74436605
 1.48520167 1.26870412 0.9219634  1.02609914 0.93732249 0.66719798
 0.08219981 1.76435313 0.45591146]
3.728244425455729
surface.crs digitized


/home/ma7/.conda/envs/cedalion/lib/python3.11/site-packages/xarray/core/variable.py:315: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)


[[181.585419  72.619293 523.590393]
 [170.857559 157.88739  523.812012]
 [181.301056  72.322266 523.730469]
 ...
 [-45.646057 139.922272 582.130432]
 [ 54.036041  -0.995697 444.60611 ]
 [ -6.470909 201.673172 444.291443]]
[[173.067749 152.878815 524.397705]
 [171.551178 156.72226  523.730469]
 [173.665268 152.718643 523.777771]
 ...
 [191.205231  50.322266 444.530457]
 [191.330841  50.72226  444.530457]
 [192.774017  53.522278 444.530457]]
O (0.11, 0.21, 0.7, 1.0)
[2.10854478e-01 1.53858763e+00 1.52551271e+00 1.41524931e+00
 8.21636977e-01 5.16551833e-01 7.68925156e-01 1.96732163e-01
 3.29924302e-01 6.12713638e-01 7.04725662e-01 9.44294724e-01
 1.44471834e+00 1.38945464e+00 6.89406622e-04 1.34719176e+00
 8.50270400e-01 7.23133865e-01 1.77280564e-01 1.55519228e+00
 4.36650376e-01]
3.536966567266555
surface.crs digitized
stickers detected -- original: 21, anonymized: 21


/home/ma7/.conda/envs/cedalion/lib/python3.11/site-packages/xarray/core/variable.py:315: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)


## 3. Nearest-neighbour matching and per-subject stats

These are exactly the numbers notebook 68 writes into
`coreg_invariance.csv`. Confirm they line up with that CSV row before
trusting the visualization in section 4.


In [4]:
def match_by_nn(a, b):
    tree = KDTree(b)
    _, idx = tree.query(a)
    return idx

idx = match_by_nn(c_orig, c_anon)
c_anon_matched = c_anon[idx]
n_anon_matched = n_anon[idx]
scalp_anon_matched = c_anon_matched - L * n_anon_matched

sticker_dev = np.linalg.norm(c_orig - c_anon_matched, axis=1)
scalp_dev = np.linalg.norm((c_orig - L * n_orig) - scalp_anon_matched, axis=1)

print(f'sticker centre deviation [mm]:')
print(f'  mean   = {sticker_dev.mean():.4f}')
print(f'  median = {np.median(sticker_dev):.4f}')
print(f'  max    = {sticker_dev.max():.4f}')
print(f'scalp-projected deviation [mm]:')
print(f'  mean   = {scalp_dev.mean():.4f}')
print(f'  median = {np.median(scalp_dev):.4f}')
print(f'  max    = {scalp_dev.max():.4f}')

unique_anon, counts = np.unique(idx, return_counts=True)
collisions = counts[counts > 1]
print(f'NN-match diagnostic: {len(unique_anon)} unique anon targets for '
      f'{len(idx)} originals; max collisions on a single anon = '
      f'{int(collisions.max()) if len(collisions) else 1}')


sticker centre deviation [mm]:
  mean   = 0.3845
  median = 0.2414
  max    = 1.5088
scalp-projected deviation [mm]:
  mean   = 0.3654
  median = 0.2509
  max    = 1.3907
NN-match diagnostic: 21 unique anon targets for 21 originals; max collisions on a single anon = 1


## 4. Side-by-side detections

Two linked PyVista panels. Left: original. Right: anonymized.

- **red spheres** -- raw `ColoredStickerProcessor` detections
  (sticker centres on the cap surface)
- **green spheres** -- scalp-projected positions (centre - 22.6 mm
  along the surface normal); these are what the validator's
  `scalp_dev` actually compares
- **cyan labelled spheres** -- five 10-20 landmarks for orientation
- **arrows** -- surface normals at each detected sticker

`link_views()` ties the cameras so rotating one panel rotates the
other. Same number of red spheres in both panels and matching
spatial layout = validator is doing what it claims.


In [5]:
pvplt = pv.Plotter(shape=(1, 2), window_size=(1600, 800))

for col, (label, surface, centres, scalp, normals) in enumerate([
    ('Original', surface_orig, centres_orig, scalp_orig, normals_orig),
    ('Anonymized', surface_anon, centres_anon, scalp_anon, normals_anon),
]):
    pvplt.subplot(0, col)
    pvplt.add_text(f'{label} (Subject{SUBJECT}) -- {len(centres)} stickers',
                   font_size=10)
    vbx.plot_surface(pvplt, surface, opacity=1.0)
    vbx.plot_labeled_points(pvplt, centres, color='red')
    vbx.plot_labeled_points(pvplt, scalp, color='green')
    vbx.plot_labeled_points(pvplt, landmarks, color='cyan', show_labels=True)
    vbx.plot_vector_field(pvplt, centres, normals)

pvplt.link_views()
pvplt.show()


Widget(value='<iframe src="http://localhost:45735/index.html?ui=P_0x7fd648682b50_0&reconnect=auto" class="pyvi…

## 5. NN-matching diagnostic

For each original sticker centre (red), draw a white line to the
nearest anon sticker centre (blue) the validator paired it with. A
healthy match looks like coincident red/blue dots and invisible
lines. A degenerate matching looks like a fan of long white lines
converging on a single blue dot -- that would mean `sticker_mean_mm`
in section 3 is averaging wrong pairs.


In [6]:
pvplt = pv.Plotter(shape=(1, 2), window_size=(1600, 800))

for col, (label, surface) in enumerate([
    ('Original surface + matches', surface_orig),
    ('Anonymized surface + matches', surface_anon),
]):
    pvplt.subplot(0, col)
    pvplt.add_text(f'{label} (Subject{SUBJECT})', font_size=10)
    vbx.plot_surface(pvplt, surface, opacity=0.5)
    vbx.plot_labeled_points(pvplt, centres_orig, color='red')
    pvplt.add_points(c_anon_matched, color='blue', point_size=12,
                     render_points_as_spheres=True)
    for a, b in zip(c_orig, c_anon_matched):
        if np.linalg.norm(a - b) > 1e-6:
            pvplt.add_mesh(pv.Line(a, b), color='white', line_width=2)

pvplt.link_views()
pvplt.show()


Widget(value='<iframe src="http://localhost:45735/index.html?ui=P_0x7fd643778550_1&reconnect=auto" class="pyvi…

## 6. What to look for

**Validator passes if all of:**

- Sticker counts in section 2 differ by 0-2 between original and
  anonymized.
- `sticker_mean_mm` in section 3 is sub-millimetre (the cap sits
  outside the deletion mask, so identical input -> identical output
  up to numerical noise from `ColoredStickerProcessor` clustering).
- Section 4 panels show the same red sticker pattern in both
  panels, no red spheres on the deleted face area.
- Section 5 NN-match diagnostic shows no white lines (or only very
  short ones), and `max collisions on a single anon` is 1.

**Validator is suspect if any of:**

- Sticker counts differ by more than 2.
- `sticker_mean_mm` exceeds a few mm despite the cap being outside
  the mask.
- Long white fan-out lines in section 5 (degenerate NN collapse).
- A red sphere appears on the anonymized scan in a region where the
  original mesh has no sticker.
